# GMM estimator scaling: VE and VP schedules

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import functools

from dem.models.components.noise_schedules import GeometricNoiseSchedule, SNRTVNoiseSchedule, exp_inverse_sigmoid_snr_sch, constant_tv_sch, fm_tv_sch 
from dem.models.components.clipper import Clipper
from dem.models.components.cvsi import get_a2_b2
from dem.models.components.sdes import ReverseSDE
from dem.models.components.sde_integration import integrate_sde
from dem.models.components.optimal_transport import wasserstein
from estimators import score_from_samples

from style import PAPER_COLORS as EST_COLOR

torch.set_default_dtype(torch.float64)                 # float64: clean unit tests, cheap here
device = "cuda" if torch.cuda.is_available() else "cpu"

# ================================ FLAGS ================================
SCHEDULE = "VP"        # "VE" or "VP"
TOY      = False        # True: 2-D / few modes (CPU smoke). False: high-dim / many modes (GPU).
SMOKE    = False        # tiny sizes for a laptop; set False for full runs on a node.
CLIP_TSI = True        # clip ONLY the TSI branch in section B.
                       # Section A is left unclipped to expose the raw identity behaviour.
## GMM:
ISOTROPIC = False
NORMALIZE = True

## Schedule specifics
GEN_START_T = 0.99
snr_sch = functools.partial(
        exp_inverse_sigmoid_snr_sch, slope=2.0, shift=0.0
    )
#tv_sch = functools.partial(constant_tv_sch, scale=1.0)
tv_sch = functools.partial(fm_tv_sch, slope=2.0, shift=0.0)
# ======================================================================


DM = dict(toy=dict(D=2, M=5), full=dict(D=20, M=20))["toy" if TOY else "full"]
CFG = dict(
    smoke=dict(B=96,  NT=8,  K_EXACT=1024, N_GEN=96,  T_STEPS=60, K_GEN=16, N_SEEDS=2),
    full =dict(B=512, NT=16, K_EXACT=1024, N_GEN=512, T_STEPS=64, K_GEN=32, N_SEEDS=1),
)["smoke" if SMOKE else "full"]

torch.manual_seed(0)
print(f"torch {torch.__version__} | device {device} | SCHEDULE={SCHEDULE} TOY={TOY} "
      f"SMOKE={SMOKE} CLIP_TSI={CLIP_TSI}")
print("dims:", DM, "| sizes:", CFG)


## 0. Normalized GMM energy with (a2, b2)-general diffusion helpers

In [ ]:
_LOG2PI = float(np.log(2 * np.pi))

class GMMEnergy:
    # p(x) = sum_i w_i N(mu_i, Sigma_i); is_molecule=False. Normalized at construction.
    is_molecule = False
    n_particles = None
    n_spatial_dim = None

    def __init__(self, means, covs, weights, normalize=True, isotropic=False):
        w = weights / weights.sum()
        if isotropic == "components":
            # Spherical components only; this does NOT make the mixture isotropic:
            # the per-dim variance spread is dominated by the scatter of the means.
            # Use isotropic=True for VP.
            _D = means.shape[-1]
            _v = torch.diagonal(covs, dim1=-2, dim2=-1).sum(-1) / _D
            covs = _v[:, None, None] * torch.eye(_D, dtype=covs.dtype, device=covs.device)
        elif isotropic:
            # Whiten the MIXTURE so its total covariance is exactly I (mean-scatter term
            # included) -- the sense VP needs, since constant TV preserves a^2 Var+b^2
            # per direction, which a scalar whitening only achieves on isotropic data.
            _mu = (w[:, None] * means).sum(0)
            _d = means - _mu
            _C = (w[:, None, None] * (covs + torch.einsum('md,me->mde', _d, _d))).sum(0)
            _ev, _evec = torch.linalg.eigh(_C)
            _W = _evec @ torch.diag(_ev.clamp_min(1e-12).rsqrt()) @ _evec.T
            means = (means - _mu) @ _W
            covs = _W @ covs @ _W
            normalize = False           # already exactly unit-variance
        if normalize:
            # scalar (isotropic) whitening to mean~0, per-dim var~1 (reference-style)
            mix_mean = (w[:, None] * means).sum(0)                       # (D,)
            diff = means - mix_mean
            mix_cov = (w[:, None, None] *
                       (covs + torch.einsum('md,me->mde', diff, diff))).sum(0)
            mix_std = torch.sqrt(torch.trace(mix_cov) / means.shape[-1])
            means = (means - mix_mean) / mix_std
            covs = covs / mix_std ** 2
        self.means = means
        self.covs = covs
        self.w = w
        self.M, self.D = means.shape
        self.Prec = torch.linalg.inv(covs)
        self.logdet = torch.logdet(covs)
        self.L = torch.linalg.cholesky(covs)

    def _eye(self, ref):
        return torch.eye(self.D, dtype=ref.dtype, device=ref.device)

    def _log_comp(self, x):
        d = x.unsqueeze(-2) - self.means
        maha = torch.einsum('...md,mde,...me->...m', d, self.Prec, d)
        return torch.log(self.w) - 0.5 * self.logdet - 0.5 * self.D * _LOG2PI - 0.5 * maha

    def __call__(self, x):
        return torch.logsumexp(self._log_comp(x), dim=-1)

    def sample(self, n):
        comp = torch.multinomial(self.w, n, replacement=True)
        z = torch.randn(n, self.D, dtype=self.means.dtype, device=self.means.device)
        return self.means[comp] + torch.einsum('nde,ne->nd', self.L[comp], z)

    def nearest_mode(self, x):
        return (x[:, None, :] - self.means[None]).norm(dim=-1).argmin(-1)

    # ---- analytic diffusion helpers for kernel N(sqrt(a2) x0, b2 I); a2,b2 scalars ----
    def _mlr(self, xt, a2, b2):                    # log responsibilities of marginal q_t
        am = torch.sqrt(a2)
        Mi = a2 * self.covs + b2 * self._eye(xt)
        d = xt.unsqueeze(-2) - am * self.means
        maha = torch.einsum('bmd,mde,bme->bm', d, torch.linalg.inv(Mi), d)
        return torch.log(self.w) - 0.5 * torch.logdet(Mi) - 0.5 * self.D * _LOG2PI - 0.5 * maha

    def marginal_score(self, xt, a2, b2):          # s*(xt) = grad log q_t
        am = torch.sqrt(a2)
        Mi = a2 * self.covs + b2 * self._eye(xt)
        Pi = torch.linalg.inv(Mi)
        d = xt.unsqueeze(-2) - am * self.means
        r = torch.softmax(self._mlr(xt, a2, b2), dim=-1)
        comp_score = -torch.einsum('mde,bme->bmd', Pi, d)
        return torch.einsum('bm,bmd->bd', r, comp_score)

    def sample_posterior_exact(self, xt, a2, b2, K):
        am = torch.sqrt(a2)
        r = torch.softmax(self._mlr(xt, a2, b2), dim=-1)                 # (B,M)
        Ci = torch.linalg.inv(self.Prec + (a2 / b2) * self._eye(xt))     # (M,D,D)
        rhs = torch.einsum('mde,me->md', self.Prec, self.means)[None] + am * xt[:, None] / b2
        mi = torch.einsum('mde,bme->bmd', Ci, rhs)                       # (B,M,D)
        B = xt.shape[0]
        Lc = torch.linalg.cholesky(Ci)
        comp = torch.multinomial(r, K, replacement=True)                 # (B,K)
        z = torch.randn(K, B, self.D, dtype=xt.dtype, device=xt.device)
        m_sel = torch.gather(mi, 1, comp.unsqueeze(-1).expand(-1, -1, self.D)).permute(1, 0, 2)
        return m_sel + torch.einsum('kbde,kbe->kbd', Lc[comp.T], z)

## 1. Build the GMM and schedule

In [ ]:
D, M = DM["D"], DM["M"]
torch.manual_seed(0)
raw_means = torch.randn(M, D) * 2.0
A = torch.randn(M, D, D) * 0.35 / np.sqrt(D)
raw_covs = torch.einsum('mde,mfe->mdf', A, A) + 0.4 * torch.eye(D)
raw_w = torch.rand(M) + 0.3
# ISOTROPIC=True forces spherical components. VP whitens with ONE scalar, exact only
# for isotropic data; with anisotropic components a2*Var+b2 holds on average, not
# per-direction. Flip to isolate that effect.
gmm = GMMEnergy(raw_means.to(device), raw_covs.to(device), raw_w.to(device),
                normalize=NORMALIZE, isotropic=ISOTROPIC)

if SCHEDULE == "VE":
    noise = GeometricNoiseSchedule(sigma_min=0.0001, sigma_max=10.0)   # h(1)=25 >> var 1
    t_lo, t_hi = 0.05, 1.0
    prior_std = float(noise.h(torch.tensor(1.0)).sqrt())
    gen_start, gen_end = 1.0, 0.0
elif SCHEDULE == "VP":
    noise = SNRTVNoiseSchedule(snr_sch=snr_sch, tv_sch=tv_sch)                                    # prior N(0, I)
    t_lo, t_hi = noise.t_min, noise.t_max
    prior_std = noise.prior_std()
    gen_start, gen_end = noise.t_max, noise.t_min
else:
    raise ValueError(SCHEDULE)

# GEN_START_T: override the reverse-SDE start time (None = schedule default).
# Under VE the prior depends on the start time, so rescale it.
if GEN_START_T is not None:
    gen_start = float(GEN_START_T)
    if SCHEDULE == 'VE':
        prior_std = float(noise.h(torch.tensor(gen_start)).sqrt())

tgrid = torch.linspace(t_lo, t_hi, CFG["NT"], device=device)

def ab_of(t):
    # Scalar (a2, b2) at scalar/uniform time t (returns 0-dim tensors).
    a2, b2 = get_a2_b2(torch.as_tensor(t, device=device).reshape(1), noise)
    return a2.reshape(()), b2.reshape(())

# TSI-only clipper (norm-clip on the energy-gradient branch). Report trigger rate later.
clipper = Clipper(True, False, max_score_norm=100.0, min_log_reward=None)

_smp = gmm.sample(4000)
print(f"SCHEDULE={SCHEDULE}  D={D} M={M}  prior_std={prior_std:.3f}")
print(f"data per-dim std ~ {_smp.std(0).mean().item():.2f}  (normalized ~1)")
print(f"a2,b2 at t_lo={t_lo:.3g}: {tuple(round(float(v),4) for v in ab_of(t_lo))}   "
      f"at t_hi={t_hi:.3g}: {tuple(round(float(v),4) for v in ab_of(t_hi))}")
print(f"mode occupancy of fresh sample: "
      f"{torch.bincount(gmm.nearest_mode(_smp), minlength=M).float().div(len(_smp)).cpu().numpy().round(2)}")

## A. Score MSE against `s*` with closed-form posterior draws

In [ ]:
K_EXACT, B = CFG["K_EXACT"], CFG["B"]
torch.manual_seed(1)
x0_data = gmm.sample(B)
rel_err = {e: [] for e in ["tsi", "dsi", "cvsi"]}
for t in tgrid:
    a2, b2 = ab_of(t)
    xt = torch.sqrt(a2) * x0_data + torch.sqrt(b2) * torch.randn_like(x0_data)
    s_star = gmm.marginal_score(xt, a2, b2)
    x0e = gmm.sample_posterior_exact(xt, a2, b2, K_EXACT)
    w = torch.full((K_EXACT, B, 1), 1.0 / K_EXACT, device=device)
    h_t = torch.full((K_EXACT, B, 1), float(b2), device=device)
    tt = t.repeat(B)
    for e in ["tsi", "dsi", "cvsi"]:
        s = score_from_samples(tt, xt, x0e, w, h_t, gmm, noise, estimator=e)
        num = (s - s_star).pow(2).sum(-1).sqrt()
        den = s_star.pow(2).sum(-1).sqrt().clamp_min(1e-9)
        rel_err[e].append((num / den).mean().item())

plt.figure(figsize=(6.6, 4))
for e in ["tsi", "dsi", "cvsi"]:
    plt.plot(tgrid.cpu(), rel_err[e], "o-", color=EST_COLOR[e], label=e.upper())
plt.yscale("log"); plt.xlabel("t"); plt.ylabel(r"rel. error $\|\hat s-s^\star\|/\|s^\star\|$")
plt.title(f"A. Identities vs $s^\\star$, exact posterior draws, {SCHEDULE} (d={D}, K={K_EXACT})")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
for e in ["tsi", "dsi", "cvsi"]:
    print(f"{e.upper():4s} rel-err: t={float(tgrid[0]):.3g} -> {rel_err[e][0]:.2e}   "
          f"t={float(tgrid[-1]):.3g} -> {rel_err[e][-1]:.2e}")
print("\nExpect CVSI <= min(TSI,DSI) at every t. Under VP, TSI grows at high t (1/a blow-up).")

## B. End-to-end reverse SDE, seed-averaged

In [ ]:
N_GEN, T_STEPS, K_GEN, N_SEEDS = CFG["N_GEN"], CFG["T_STEPS"], CFG["K_GEN"], CFG["N_SEEDS"]
mu_mix = (gmm.w[:, None] * gmm.means).sum(0)
diff_mix = gmm.means - mu_mix
Sigma_mix = (gmm.w[:, None, None] *
             (gmm.covs + torch.einsum('md,me->mde', diff_mix, diff_mix))).sum(0)
ref = gmm.sample(2000)

def make_score_fn(est):
    clip = clipper if CLIP_TSI else None
    def exact_score_fn(t, x):
        a2, b2 = ab_of(t.reshape(-1)[0])
        x0e = gmm.sample_posterior_exact(x, a2, b2, K_GEN)
        w = torch.full((K_GEN, x.shape[0], 1), 1.0 / K_GEN, device=device)
        h_t = torch.full((K_GEN, x.shape[0], 1), float(b2), device=device)
        return score_from_samples(t, x, x0e, w, h_t, gmm, noise,
                                  estimator=est, clipper_tsi=clip)
    return exact_score_fn

def generate(est, seed):
    torch.manual_seed(seed)
    sde = ReverseSDE(make_score_fn(est), noise)
    x_init = torch.randn(N_GEN, D, device=device) * prior_std
    traj = integrate_sde(sde, x_init, num_integration_steps=T_STEPS, energy_function=gmm,
                         diffusion_scale=1.0, no_grad=True,
                         start_time=gen_start, end_time=gen_end)
    return traj[-1].detach()

rows, gen_cache = [], {}
for e in ["tsi", "dsi", "cvsi"]:
    me, ce, w2, nm = [], [], [], []
    for s in range(N_SEEDS):
        g = generate(e, seed=s)
        if s == 0:
            gen_cache[e] = g
        me.append((g.mean(0) - mu_mix).norm().item())
        ce.append((torch.cov(g.T) - Sigma_mix).norm().item())
        w2.append(wasserstein(g, ref[:N_GEN], power=2))
        nm.append(len(torch.unique(gmm.nearest_mode(g))))
    agg = lambda v: (float(torch.tensor(v, dtype=torch.float64).mean()),
                     float(torch.tensor(v, dtype=torch.float64).std()) if len(v) > 1 else 0.0)
    rows.append((e.upper(), agg(me), agg(ce), agg(w2), agg(nm)))

In [ ]:
print(f"SCHEDULE={SCHEDULE}  (mean +- std over {N_SEEDS} seeds; exact posterior; TSI-clip={'on' if CLIP_TSI else 'off'})")
print(f"{'ident':6s}{'mean_err':>14s}{'cov_err':>14s}{'W2':>14s}{'modes/'+str(M):>11s}")
for e, me, ce, w2, nm in rows:
    print(f"{e:6s}"
          f"{me[0]:9.3f}+-{me[1]:4.2f}"
          f"{ce[0]:9.3f}+-{ce[1]:4.2f}"
          f"{w2[0]:9.3f}+-{w2[1]:4.2f}"
          f"{nm[0]:7.1f}+-{nm[1]:3.1f}")
print(f"\ntrue mixture mean norm={mu_mix.norm().item():.3f}  ('modes' near {M} = full coverage)")


### B. PCA projection of the generated samples

In [ ]:
ref_c = ref - ref.mean(0)
_, _, Vh = torch.linalg.svd(ref_c, full_matrices=False)
P = Vh[:2].T
def proj(x): return ((x - ref.mean(0)) @ P).cpu()
means_p = proj(gmm.means); rp = proj(ref[:1500])

ests = ["tsi", "dsi", "cvsi"]
fig, ax = plt.subplots(1, len(ests), figsize=(4.8*len(ests), 4.6), sharex=True, sharey=True)
for a, e in zip(ax, ests):
    a.scatter(rp[:, 0], rp[:, 1], s=6, alpha=0.15, c="k", label="true")
    g = proj(gen_cache[e])
    a.scatter(g[:, 0], g[:, 1], s=10, alpha=0.5, c=EST_COLOR[e], label=e.upper())
    a.scatter(means_p[:, 0], means_p[:, 1], marker="*", s=220, c="red", edgecolor="k", zorder=5)
    a.set_title(e.upper()); a.set_xlabel("PC1"); a.legend(fontsize=8)
ax[0].set_ylabel("PC2")
plt.suptitle(f"PCA: true GMM vs generated samples ({SCHEDULE}, d={D})", y=1.02)
plt.tight_layout(); plt.show()
